In [1]:
import numpy as np
import os
import json
from typing import List, Dict, Any
import os

In [2]:
os.environ['http_proxy']="http://proxy61.iitd.ac.in:3128"
os.environ['https_proxy']="http://proxy61.iitd.ac.in:3128"

In [3]:
split = True

## Reading the ranked results and the queries

In [4]:
# model parameters
run_number = "without_nan_with_space_remv_p2_n2_run_1_balanced_2"
m_min_tables = 10
m_min_rows = 8
m_max_rows = 12
m_min_cols = 10
m_max_cols = 25

# corpus parameters
split_number = "without_nan_with_space_remv_run_1"
min_tables = 10
min_rows = 8
max_rows = 12
min_cols = 10
max_cols = 25


In [5]:
m_parm_setting = f'mint_{m_min_tables}_minr_{m_min_rows}_maxr_{m_max_rows}_minc_{m_min_cols}_maxc_{m_max_cols}'
parm_setting = f'mint_{min_tables}_minr_{min_rows}_maxr_{max_rows}_minc_{min_cols}_maxc_{max_cols}'
model_setting = "batch_4_epoch_8_lr_3e-05"

result_path = f"../../output/spider_res/default_{model_setting}_{run_number}_{m_parm_setting}_{parm_setting}_{split_number}/gemini-2.5-flash/"
query_path = "../../dataset/spider/dev.json"
dataset_path = f"../../dataset/spider/processing/corpus_split/{split_number}_{parm_setting}"

# coverage_path = f"../../dataset/spider/processing/corpus_coverage/{split_number}"

In [6]:
# reading the ground-truth tables 

files_corpus = []

# checking the number of files in query_tables path
if os.path.exists(dataset_path) and os.path.isdir(dataset_path):
    files_corpus = [f for f in os.listdir(dataset_path) if os.path.isfile(os.path.join(dataset_path, f))]
else:
    print(f"Folder not found at {dataset_path}")

# files_corpus

In [7]:
groud_truth_table_list = []

# Read the data
if len(files_corpus) != 0:
    for f in files_corpus:
        
        with open(os.path.join(dataset_path,f), 'r') as fp:
            json_data = json.load(fp)
        data_keys = json_data.keys()

        for data_key in data_keys:
            if split == True:
                data_key = data_key.strip().split('__')[0]
            data_key = data_key.strip().split('table-')[1]
            groud_truth_table_list.append(data_key.lower())
            

In [8]:
# groud_truth_table_list

In [9]:
# reading the json file
with open(query_path, 'r') as fp:
    data = json.load(fp)
    

In [10]:
# extracting the query id and relevant table from the json
query_table = {}
for d in data:
    query_table[d['id']+1] = []

    for i in d['rel_schema']:
        query_table[d['id']+1].append((i.split("(")[0].strip()).replace(".","-").replace(" ", "_"))
        

In [11]:
# query_table

In [12]:
## cleaning query table. 
# Removing the queryid whose tables are not present in ground-truth-tables
# Removing the table names that are not present in the ground-truth-tables

query_table_clean = {}

queries_with_no_tables = []

for k in query_table.keys():
    val = []

    for v in query_table[k]:
        if v.lower() in groud_truth_table_list:
            val.append(v.lower())
    
    if len(val)>0:
        query_table_clean[k] = val
    else:
        # print(k, query_table[k])
        queries_with_no_tables.append(int(k))
        

In [13]:
# print(len(query_table_clean))

In [14]:
def process_data(data, pt):
    data  = data.strip().split("\n")
    
    fp = open(f'{pt}', 'w')

    for d in data:
        tmp = d.strip().split(":")

        sc = tmp[0].strip()
        nm = tmp[1].strip()

        fp.write(f'{sc} : ')

        nm_t = nm.strip().split('table-')
        for v in nm_t:
            if v.strip() != "":
                fp.write(f'table-{v.strip()} ')
        
        fp.write("\n")

    fp.close()
    

In [15]:
# Reading the ranked results
files = [f for f in os.listdir(result_path) if os.path.isfile(os.path.join(result_path, f))]

results = {}

for f in files:
    with open(os.path.join(result_path, f), 'r') as fp:
        data = fp.read()

    # process_data(data, os.path.join(result_path, f))

    data = data.strip().split("\n")

    res = {}

    for idx, d in enumerate(data):
        tmp = d.strip().split(":")

        # sc is the similarity score
        sc = float(tmp[0].strip())

        # nm is the name of the tables [it is possible that nm contain multiple tables]
        nm = tmp[1].strip()
        # here nm_t is the list of all the tables for the given sc
        nm_t = nm.strip().split(' ')

        for n in nm_t:

            if split == True:

                # since we are using split which has additional __{idx} so we have to remove it
                
                n = n.strip().split("__")[0]
                n = n.strip()

                if n.lower() in res.keys():
                    res[n.lower()].append([idx+1, sc])
                else:
                    res[n.lower()] = [[idx+1, sc]]
            else:
                res[n.lower()] = [idx+1,sc]
    
    results[f] = res

In [16]:
def maxSubTableSelection(subtable_rank):
    table_max_dict = {}

    for subtable in subtable_rank.keys():
        sub_table_name = subtable
        sub_table_score = []

        for sc in subtable_rank[subtable]:
            sub_table_score.append(sc[1])

        sims = np.asarray(sub_table_score, dtype=np.float32)
        table_max_dict[sub_table_name] = np.max(sims)

    return table_max_dict


def avgSubTableSelectionAtTopK(subtable_rank, k=5):
    table_max_dict = {}

    for subtable in subtable_rank.keys():
        sub_table_name = subtable
        sub_table_score = []

        for sc in subtable_rank[subtable]:
            sub_table_score.append(sc[1])

        sims = np.asarray(sub_table_score, dtype=np.float32)
        # print(sims.size)

        # handling the edge case
        k = min(k, sims.size)

        topk_sorted = np.sort(sims)[-k:][::-1]
        table_max_dict[sub_table_name] = float(np.mean(topk_sorted))

    return table_max_dict

def softmaxAggregation(subtable_rank, alpha=15.0):
    """
    Softmax aggregation of subtable similarity scores.

    Args:
        subtable_rank (dict): list of [ rank and cosine similarities ] for each subtable for each table in the corpus
        alpha (float): temperature parameter controlling sharpness

    Returns:
        float: aggregated table-level score for each table
    """

    table_softmaxagg_dict = {}

    # now for each table we need a score
    for subtable in subtable_rank.keys():
        sub_table_name = subtable
        sub_table_score = []

        for sc in subtable_rank[subtable]:
            sub_table_score.append(sc[1])
        
        sims = np.asarray(sub_table_score, dtype=np.float32)

        # Numerical stability: subtract max before exp
        sims_scaled = alpha * sims
        sims_scaled -= np.max(sims_scaled)

        weights = np.exp(sims_scaled)
        weights /= np.sum(weights)

        table_softmaxagg_dict[sub_table_name] = float(np.sum(weights * sims))


    return table_softmaxagg_dict

In [17]:
## calculating softmax agg for each query table
## Here results is a dict of list of [idx, score]
softmax_res = {}
max_res = {}
avg_at_k_res = {}

for res in results.keys():
    softmax_res[res] = softmaxAggregation(results[res], 3.0)
    max_res[res] = maxSubTableSelection(results[res])
    avg_at_k_res[res] = avgSubTableSelectionAtTopK(results[res], 3)

In [18]:
# softmax_res['table-1-1']

In [19]:
import numpy as np

def dcg_at_k(relevances, k):
    """
    Compute DCG@k for a list of relevance labels.
    relevances: list or np.array of relevance scores (0/1)
    k: cutoff
    """
    relevances = np.asarray(relevances)[:k]
    if relevances.size == 0:
        return 0.0

    discounts = np.log2(np.arange(2, relevances.size + 2))
    gains = (2 ** relevances - 1) / discounts
    return np.sum(gains)


def ndcg_at_k(y_true, y_score, k):
    """
    Compute nDCG@k.
    y_true: ground-truth relevance labels (0/1)
    y_score: predicted scores (higher = more relevant)
    k: cutoff
    """
    # Rank documents by predicted score
    order = np.argsort(y_score)[::-1]
    y_true_sorted = np.asarray(y_true)[order]

    # DCG@k
    dcg = dcg_at_k(y_true_sorted, k)

    # Ideal DCG@k (sort by true relevance)
    ideal_order = np.argsort(y_true)[::-1]
    y_true_ideal = np.asarray(y_true)[ideal_order]
    idcg = dcg_at_k(y_true_ideal, k)

    if idcg == 0.0:
        return 0.0

    return dcg / idcg

In [20]:
import numpy as np

def recall_at_k(y_true, y_score, k):
    """
    Compute Recall@k for a single query.

    y_true: list or np.array of ground-truth relevance labels (0/1)
    y_score: list or np.array of predicted scores (higher = more relevant)
    k: cutoff
    """
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # Number of relevant items in ground truth
    num_relevant = np.sum(y_true)
    # print(num_relevant, end = ", ")

    # Edge case: no relevant documents
    if num_relevant == 0:
        return 0.0

    # Get indices of top-k predicted scores
    top_k_indices = np.argsort(y_score)[::-1][:k]

    # Count relevant items in top-k
    num_relevant_in_top_k = np.sum(y_true[top_k_indices])

    # print(num_relevant_in_top_k)

    return num_relevant_in_top_k / num_relevant


In [21]:
import numpy as np

def mrr_at_k(y_true, y_score, k):
    """
    Compute MRR@k for a single query.

    y_true: list or np.array of ground-truth relevance labels (0/1)
    y_score: list or np.array of predicted scores (higher = more relevant)
    k: cutoff
    """
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # Rank documents by predicted score (descending)
    ranked_indices = np.argsort(y_score)[::-1][:k]

    # Find the rank of the first relevant document
    for rank, idx in enumerate(ranked_indices, start=1):
        if y_true[idx] == 1:
            return 1.0 / rank

    # No relevant document found in top-k
    return 0.0

In [22]:
# calculating the recall at k
top_k = [3, 5, 10, 20]

recall_list = []
ndcg_list = []
mrr_list = []

for r in top_k:
    recall = 0
    ndcg = 0
    mrr = 0

    for k in softmax_res.keys():
        
        # print(k)
        
        ## sort the ranking of the tables
        sorted_rank = dict(sorted(softmax_res[k].items(), key=lambda item: item[1], reverse=True))
        # print(sorted_rank)
        # print(sorted_rank)

        ## if a query does not have any valid answer then skip
        if int(k.strip().split("-")[-1].strip()) in queries_with_no_tables:
            continue

        ## Ground truth ranking of the tables
        result_set = query_table_clean[int(k.strip().split("-")[-1].strip())]
        # print(result_set)

        ## making y_true and y_score list
        y_true = []
        y_score = []

        for t_name in sorted_rank.keys():
            y_score.append(sorted_rank[t_name])

            t_name_tmp = t_name.strip().split("table-")[-1].strip()
            # print(t_name_tmp)
            if t_name_tmp.lower() in result_set:
                y_true.append(1)
            else:
                y_true.append(0)

        # if sum(y_true) == 0:
        #     print(k, result_set)

        ## calculating ndcg@r
        # print(f"nDCG@{r}:", ndcg_at_k(y_true, y_score, r))
        ndcg += ndcg_at_k(y_true, y_score, r)
        
        ## calculating recall@r
        # print(f"Recall@{r}:", recall_at_k(y_true, y_score, r))
        recall += recall_at_k(y_true, y_score, r)

        ## calculating mrr@r ## It is not useful as we have multiple tables for one query
        # print(f"MRR@{r}:", mrr_at_k(y_true, y_score, r))
        mrr += mrr_at_k(y_true, y_score, r)


    ndcg_list.append(ndcg)
    recall_list.append(recall)
    mrr_list.append(mrr)

In [23]:
# print(f'{model_setting}__{run_number}_{m_parm_setting}__{split_number}_{parm_setting}__softmax')

# ## Printing the result
# for k, ndcg, recall in zip(top_k, ndcg_list, recall_list):
#     print(f'nDCG@{k}: {ndcg/(len(softmax_res.keys())-len(queries_with_no_tables))} \t recall@{k}: {recall/(len(softmax_res.keys())-len(queries_with_no_tables))}')

In [24]:
# calculating the recall at k
top_k = [3, 5, 10, 20]

recall_list = []
ndcg_list = []
mrr_list = []

for r in top_k:
    recall = 0
    ndcg = 0
    mrr = 0

    for k in max_res.keys():
        
        ## sort the ranking of the tables
        sorted_rank = dict(sorted(max_res[k].items(), key=lambda item: item[1], reverse=True))
        # print(sorted_rank)
        # print(sorted_rank)

        ## if a query does not have any valid answer then skip
        if int(k.strip().split("-")[-1].strip()) in queries_with_no_tables:
            continue

        ## Ground truth ranking of the tables
        result_set = query_table_clean[int(k.strip().split("-")[-1].strip())]
        # print(result_set)

        ## making y_true and y_score list
        y_true = []
        y_score = []

        for t_name in sorted_rank.keys():
            y_score.append(sorted_rank[t_name])

            t_name_tmp = t_name.strip().split("table-")[-1].strip()
            if t_name_tmp in result_set:
                y_true.append(1)
            else:
                y_true.append(0)

        ## calculating ndcg@r
        # print(f"nDCG@{r}:", ndcg_at_k(y_true, y_score, r))
        ndcg += ndcg_at_k(y_true, y_score, r)
        
        ## calculating recall@r
        # print(f"Recall@{r}:", recall_at_k(y_true, y_score, r))
        recall += recall_at_k(y_true, y_score, r)

        ## calculating mrr@r ## It is not useful as we have multiple tables for one query
        # print(f"MRR@{r}:", mrr_at_k(y_true, y_score, r))
        mrr += mrr_at_k(y_true, y_score, r)


    ndcg_list.append(ndcg)
    recall_list.append(recall)
    mrr_list.append(mrr)

In [25]:
# print(f'{model_setting}__{run_number}_{m_parm_setting}__{split_number}_{parm_setting}__max')

# ## Printing the result
# for k, ndcg, recall in zip(top_k, ndcg_list, recall_list):
#     print(f'nDCG@{k}: {ndcg/(len(max_res.keys())-len(queries_with_no_tables))} \t recall@{k}: {recall/(len(max_res.keys())-len(queries_with_no_tables))}')

In [26]:
# calculating the recall at k
top_k = [3, 5, 10, 20]

recall_list = []
ndcg_list = []
mrr_list = []

for r in top_k:
    recall = 0
    ndcg = 0
    mrr = 0

    for k in avg_at_k_res.keys():
        
        ## sort the ranking of the tables
        sorted_rank = dict(sorted(avg_at_k_res[k].items(), key=lambda item: item[1], reverse=True))
        # print(sorted_rank)
        # print(sorted_rank)

        ## if a query does not have any valid answer then skip
        if int(k.strip().split("-")[-1].strip()) in queries_with_no_tables:
            continue

        ## Ground truth ranking of the tables
        result_set = query_table_clean[int(k.strip().split("-")[-1].strip())]
        # print(result_set)

        ## making y_true and y_score list
        y_true = []
        y_score = []

        for t_name in sorted_rank.keys():
            y_score.append(sorted_rank[t_name])

            t_name_tmp = t_name.strip().split("table-")[-1].strip()
            if t_name_tmp in result_set:
                y_true.append(1)
            else:
                y_true.append(0)

        ## calculating ndcg@r
        # print(f"nDCG@{r}:", ndcg_at_k(y_true, y_score, r))
        ndcg += ndcg_at_k(y_true, y_score, r)
        
        ## calculating recall@r
        # print(f"Recall@{r}:", recall_at_k(y_true, y_score, r))
        recall += recall_at_k(y_true, y_score, r)

        ## calculating mrr@r ## It is not useful as we have multiple tables for one query
        # print(f"MRR@{r}:", mrr_at_k(y_true, y_score, r))
        mrr += mrr_at_k(y_true, y_score, r)


    ndcg_list.append(ndcg)
    recall_list.append(recall)
    mrr_list.append(mrr)

In [27]:
# print(f'{model_setting}__{run_number}_{m_parm_setting}__{split_number}_{parm_setting}__avg_at_k')

## Printing the result
for k, ndcg, recall in zip(top_k, ndcg_list, recall_list):
    # print(f'nDCG@{k}: {ndcg/(len(avg_at_k_res.keys())-len(queries_with_no_tables))} \t recall@{k}: {recall/(len(avg_at_k_res.keys())-len(queries_with_no_tables))}')
    print(f'& {recall/(len(avg_at_k_res.keys())-len(queries_with_no_tables))}', end=" ")
print()

& 0.6156121399176954 & 0.7071759259259258 & 0.7698045267489714 & 0.8245884773662552 


In [28]:
# def get_recall_k(r, result_set, retrieved_set):
#     cnt = 0
#     div = 0
    
#     # print(retrieved_set)

#     if split == True:
        
#         for i in result_set:
#             try:
#                 if retrieved_set[f'table-{i.lower()}'][0] <= r:
#                     cnt+=1
#                 div+=1
#             except:
#                 pass
        
#         if div>0:
#             return cnt/div
#         else:
#             return -1

#     else:

#         for i in result_set:

#             try:
#                 if retrieved_set[f'table-{i.lower()}'] <= r:
#                     cnt+=1
#                 div+=1
#             except:
#                 pass
        
#         if div>0:
#             return cnt/div
#         else:
#             return -1
        

In [29]:
# # computing recall
# top_k = [3, 5, 10, 20]

# recall_list = []

# for r in top_k:
#     recall = 0
#     for k in query_table_clean.keys():
#         try:
#             query_id = k
#             result_set = query_table_clean[k]
#             retrieved_set = results[f'table-{k}-{k}']

#             # print (k)

#             tmp = get_recall_k(r, result_set, retrieved_set)
            
#             if tmp != -1:
#                 recall += tmp
#         except:
#             print(k)
    
#     recall_list.append(recall)
    

In [30]:
# for k, r in zip(top_k, recall_list):
#     print(f'recall@{k} : {r/len(query_table.keys())}')
    